# BiRNN + Attention — Next-Day Binary Classification

**Goal:** Predict next-day wheat futures price direction (**up / down**) using
BiRNN + Attention architecture.

**Pipeline:** Raw FRED-MD → t-code transforms → 1-month delay → forward-fill →
30 lagged prices + 31 macro features → BiRNN + Attention classifier → 5-fold CV + Optuna.

In [ ]:
!pip install optuna --quiet
!pip install keras-self-attention --quiet

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import pickle
import warnings

from abc import ABC, abstractmethod
from pathlib import Path
from tqdm.auto import tqdm

import seaborn as sns
import optuna
import tensorflow as tf

from tensorflow.keras import layers, regularizers, Input, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import TimeSeriesSplit
from keras_self_attention import SeqSelfAttention

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

random.seed(42); np.random.seed(42); tf.random.set_seed(42)
print("Imports loaded.")

In [ ]:
class BaseForecastModel(ABC):
    def __init__(self, task_type, **hp):
        self.task_type = task_type
        self.hyperparameters = hp
    @abstractmethod
    def fit(self, X, y): pass
    @abstractmethod
    def predict(self, X): pass
    @abstractmethod
    def evaluate(self, X, y): pass
    @abstractmethod
    def save(self, fp): pass
    @abstractmethod
    def load(self, fp): pass
print("BaseForecastModel defined.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/Quants ")
W18 = BASE_DIR / "Investing.com" / "US Wheat Futures Historical Data_2018.csv"
W25 = BASE_DIR / "Investing.com" / "US Wheat Futures Historical Data_2025.csv"

def load_price(p):
    df = pd.read_csv(p)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").set_index("Date")
    df["Price"] = df["Price"].astype(str).str.replace(",","",regex=False).astype(float)
    return df["Price"]

p1, p2 = load_price(W18), load_price(W25)
df_price = pd.concat([p1, p2.iloc[1:]]).pipe(
    lambda s: s[~s.index.duplicated(keep="last")]).sort_index()
df_price.name = "Price"
print(f"Wheat prices: {len(df_price)} days")

## FRED-MD: t-code transforms → 1-month delay → daily forward-fill

In [ ]:
fred_md_path = '/content/drive/MyDrive/Quants /FredMD_Dataset/2025-10-MD.csv'
tcodes_row = pd.read_csv(fred_md_path, nrows=1)
fred_raw = pd.read_csv(fred_md_path, skiprows=[1])
fred_raw['sasdate'] = pd.to_datetime(fred_raw['sasdate'], format='%m/%d/%Y')
fred_raw = fred_raw.set_index('sasdate').sort_index()

features_31 = [
    "RPI","W875RX1","CMRMTSPLx","IPFPNSS","USWTRADE","USTRADE",
    "BUSLOANS","CONSPI","S&P 500","S&P PE ratio","FEDFUNDS","TB3MS",
    "TB6MS","GS1","GS5","GS10","AAA","BAA","TB3SMFFM","TB6SMFFM",
    "T1YFFM","T5YFFM","T10YFFM","AAAFFM","BAAFFM",
    "EXSZUSx","EXJPUSx","EXUSUKx","EXCAUSx","PPICMM","UMCSENTx",
]
tcode_map = {c: int(float(tcodes_row[c].values[0])) for c in features_31}

def apply_tcode(s, tc):
    if tc==1: return s
    if tc==2: return s.diff()
    if tc==3: return s.diff().diff()
    if tc==4: return np.log(s.clip(lower=1e-10))
    if tc==5: return np.log(s.clip(lower=1e-10)).diff()
    if tc==6: return np.log(s.clip(lower=1e-10)).diff().diff()
    if tc==7: return (s/s.shift(1)-1).diff()
    return s

fs = fred_raw[features_31].copy().apply(pd.to_numeric, errors='coerce')
for c in features_31:
    fs[c] = apply_tcode(fs[c], tcode_map[c])
fred_t = fs.dropna().replace([np.inf,-np.inf], np.nan).dropna()

fred_t.index = fred_t.index + pd.DateOffset(months=1)
daily_idx = pd.date_range(fred_t.index.min(), fred_t.index.max()+pd.offsets.MonthEnd(0), freq='D')
df_macro = fred_t.reindex(daily_idx).ffill()

print(f"Transformed macro: {df_macro.shape}, daily frequency")

In [ ]:
df_cls = df_macro.join(df_price, how='inner').loc['2008-01-01':]
cols = [c for c in df_cls.columns if c!='Price'] + ['Price']
df_cls = df_cls[cols]

LB = 30
pidx = df_cls.columns.get_loc('Price')
data = df_cls.values
X, y = [], []
for i in range(LB, len(data)):
    X.append(data[i-LB:i])
    y.append(1 if data[i, pidx] > data[i-1, pidx] else 0)
X, y = np.array(X), np.array(y)

sp = int(len(X)*0.8)
X_train, y_train = X[:sp], y[:sp]
X_test,  y_test  = X[sp:], y[sp:]
dates = df_cls.index[LB:]
train_dates, test_dates = dates[:sp], dates[sp:]

print(f"X: {X.shape}  |  Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Class dist (all): Up={y.sum()} ({y.mean():.2%}), Down={len(y)-y.sum()}")

## BiRNN + Attention Classifier

Architecture: Stacked Bidirectional SimpleRNN + **SeqSelfAttention** + GlobalAveragePooling

Output: sigmoid → binary cross-entropy, balanced class weights.

In [ ]:
class BiRNNAttentionClassifier(BaseForecastModel):

    def __init__(self, task_type='classification', hidden_size=64,
                 num_layers=2, weight_decay=0.02, dropout=0.2,
                 early_stop_patience=10, epochs=100, batch_size=64,
                 learning_rate=1e-3):
        super().__init__(task_type=task_type, hidden_size=hidden_size,
                         num_layers=num_layers, weight_decay=weight_decay,
                         dropout=dropout, early_stop_patience=early_stop_patience,
                         epochs=epochs, batch_size=batch_size,
                         learning_rate=learning_rate)
        self.hidden_size=hidden_size; self.num_layers=num_layers
        self.weight_decay=weight_decay; self.dropout=dropout
        self.early_stop_patience=early_stop_patience
        self.epochs=epochs; self.batch_size=batch_size
        self.learning_rate=learning_rate
        self.model=None; self.x_scaler=None
        self.lookback=None; self.n_features=None

    def _build_model(self, seq_length, feature_dim):
        inputs = Input(shape=(seq_length, feature_dim))

        h = layers.Bidirectional(
            layers.SimpleRNN(
                units=self.hidden_size, dropout=self.dropout,
                kernel_regularizer=regularizers.l2(self.weight_decay),
                return_sequences=True,
            )
        )(inputs)

        for _ in range(self.num_layers - 1):
            h = layers.Bidirectional(
                layers.SimpleRNN(
                    units=self.hidden_size, dropout=self.dropout,
                    kernel_regularizer=regularizers.l2(self.weight_decay),
                    return_sequences=True,
                )
            )(h)

        h = SeqSelfAttention(attention_activation='sigmoid')(h)
        x = layers.GlobalAveragePooling1D()(h)
        x = layers.Dropout(self.dropout)(x)

        outputs = layers.Dense(
            1, activation='sigmoid',
            kernel_regularizer=regularizers.l2(self.weight_decay),
        )(x)
        return Model(inputs=inputs, outputs=outputs)

    def _scale_X(self, X, fit=False):
        n,lb,nf = X.shape
        flat = X.reshape(n, lb*nf)
        if fit:
            self.x_scaler = StandardScaler()
            flat = self.x_scaler.fit_transform(flat)
        else:
            flat = self.x_scaler.transform(flat)
        return flat.reshape(n,lb,nf)

    def fit(self, X_train, y_train):
        self.lookback, self.n_features = X_train.shape[1], X_train.shape[2]
        Xs = self._scale_X(X_train, fit=True)
        cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
        self.model = self._build_model(self.lookback, self.n_features)
        self.model.compile(optimizer=Adam(learning_rate=self.learning_rate),
                           loss='binary_crossentropy', metrics=['accuracy'])
        self.model.fit(Xs, y_train, epochs=self.epochs, batch_size=self.batch_size,
                       shuffle=False, class_weight={0:cw[0],1:cw[1]}, verbose=0,
                       callbacks=[
                           EarlyStopping(monitor='loss', patience=self.early_stop_patience,
                                         restore_best_weights=True, verbose=0),
                           ReduceLROnPlateau(monitor='loss', factor=0.5, patience=5,
                                             min_lr=1e-6, verbose=0)])
        p = (self.model.predict(Xs, verbose=0).ravel()>=0.5).astype(int)
        print(f"  Train Acc: {accuracy_score(y_train,p):.4f}  "
              f"F1: {f1_score(y_train,p,zero_division=0):.4f}")

    def predict(self, X):
        return self.model.predict(self._scale_X(X), verbose=0).ravel()

    def evaluate(self, X, y, th=0.5):
        pr = self.predict(X); pd_ = (pr>=th).astype(int)
        m = {'accuracy': accuracy_score(y,pd_),
             'precision': precision_score(y,pd_,zero_division=0),
             'recall': recall_score(y,pd_,zero_division=0),
             'f1': f1_score(y,pd_,zero_division=0)}
        try: m['auc_roc'] = roc_auc_score(y,pr)
        except: m['auc_roc'] = 0.5
        return m

    def save(self, fp):
        self.model.save_weights(fp+'.weights.h5')
        with open(fp+'.meta.pkl','wb') as f:
            pickle.dump({'hp':self.hyperparameters,'scaler':self.x_scaler,
                         'lb':self.lookback,'nf':self.n_features}, f)

    def load(self, fp):
        with open(fp+'.meta.pkl','rb') as f: d=pickle.load(f)
        self.x_scaler=d['scaler']; self.lookback=d['lb']; self.n_features=d['nf']
        self.model=self._build_model(self.lookback, self.n_features)
        self.model.load_weights(fp+'.weights.h5')

print("BiRNNAttentionClassifier defined.")

In [ ]:
def cv_birnn_attention_cls(X_train, y_train, **kw):
    tscv = TimeSeriesSplit(n_splits=5)
    lb, nf = X_train.shape[1], X_train.shape[2]
    fm = {k:[] for k in ['accuracy','precision','recall','f1','auc_roc','val_loss']}
    all_p, all_t = [], []

    for fold,(ti,vi) in tqdm(enumerate(tscv.split(X_train)), total=5, desc="CV"):
        Xtr, Xv = X_train[ti], X_train[vi]
        ytr, yv = y_train[ti], y_train[vi]
        sc = StandardScaler()
        Xtr_s = sc.fit_transform(Xtr.reshape(len(Xtr),-1)).reshape(len(Xtr),lb,nf)
        Xv_s  = sc.transform(Xv.reshape(len(Xv),-1)).reshape(len(Xv),lb,nf)
        cw = compute_class_weight('balanced', classes=np.array([0,1]), y=ytr)

        tmp = BiRNNAttentionClassifier(**kw)
        model = tmp._build_model(lb, nf)
        model.compile(optimizer=Adam(learning_rate=kw.get('learning_rate',1e-3)),
                       loss='binary_crossentropy', metrics=['accuracy'])
        h = model.fit(Xtr_s, ytr, validation_data=(Xv_s, yv),
                      epochs=kw.get('epochs',100), batch_size=kw.get('batch_size',64),
                      shuffle=False, class_weight={0:cw[0],1:cw[1]}, verbose=0,
                      callbacks=[
                          EarlyStopping(monitor='val_loss', patience=kw.get('early_stop_patience',10),
                                        restore_best_weights=True, verbose=0),
                          ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                                            min_lr=1e-6, verbose=0)])
        fm['val_loss'].append(min(h.history['val_loss']))
        vp = model.predict(Xv_s, verbose=0).ravel()
        vpd = (vp>=0.5).astype(int)
        fm['accuracy'].append(accuracy_score(yv,vpd))
        fm['precision'].append(precision_score(yv,vpd,zero_division=0))
        fm['recall'].append(recall_score(yv,vpd,zero_division=0))
        fm['f1'].append(f1_score(yv,vpd,zero_division=0))
        try: fm['auc_roc'].append(roc_auc_score(yv,vp))
        except: fm['auc_roc'].append(0.5)
        all_p.extend(vp); all_t.extend(yv)
        print(f"  Fold {fold+1}: Acc={fm['accuracy'][-1]:.4f} F1={fm['f1'][-1]:.4f} AUC={fm['auc_roc'][-1]:.4f}")

    return {k:np.mean(v) for k,v in fm.items()}, np.array(all_t), np.array(all_p)

## Default Hyperparameters — 5-fold CV

In [ ]:
random.seed(42); np.random.seed(42); tf.random.set_seed(42)
def_m, def_t, def_p = cv_birnn_attention_cls(X_train, y_train)

print("\n=== Default CV (5-fold avg) ===")
for k,v in def_m.items(): print(f"  {k:12s}: {v:.6f}")
print("\n" + classification_report(def_t, (def_p>=0.5).astype(int), target_names=['Down','Up']))

## Optuna Hyperparameter Tuning (25 trials, maximise F1)

In [ ]:
def optuna_obj(trial):
    p = {
        'learning_rate':      trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True),
        'hidden_size':        trial.suggest_categorical('hidden_size', [32,64,128]),
        'num_layers':         trial.suggest_int('num_layers', 1, 3),
        'weight_decay':       trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
        'dropout':            trial.suggest_float('dropout', 0.1, 0.5),
        'early_stop_patience': trial.suggest_int('early_stop_patience', 5, 15),
        'epochs':             trial.suggest_categorical('epochs', [50,100,150]),
        'batch_size':         trial.suggest_categorical('batch_size', [32,64,128]),
    }
    avg, _, _ = cv_birnn_attention_cls(X_train, y_train, **p)
    return avg['f1']

random.seed(42); np.random.seed(42); tf.random.set_seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', study_name='BiRNNAttentionClassifier')
study.optimize(optuna_obj, n_trials=25, show_progress_bar=True)

print(f"\nBest F1: {study.best_value:.6f}")
print("Best params:")
for k,v in study.best_params.items(): print(f"  {k}: {v}")

In [ ]:
random.seed(42); np.random.seed(42); tf.random.set_seed(42)
opt_m, opt_t, opt_p = cv_birnn_attention_cls(X_train, y_train, **study.best_params)

print("\n=== Optuna CV (5-fold avg) ===")
for k,v in opt_m.items(): print(f"  {k:12s}: {v:.6f}")
print("\n" + classification_report(opt_t, (opt_p>=0.5).astype(int), target_names=['Down','Up']))

## Final Test Set Evaluation

In [ ]:
random.seed(42); np.random.seed(42); tf.random.set_seed(42)
final = BiRNNAttentionClassifier(**study.best_params)
final.fit(X_train, y_train)

test_probs = final.predict(X_test)
test_preds = (test_probs >= 0.5).astype(int)

print("\n=== Test Set Results ===")
print(f"  Accuracy:  {accuracy_score(y_test, test_preds):.6f}")
print(f"  Precision: {precision_score(y_test, test_preds, zero_division=0):.6f}")
print(f"  Recall:    {recall_score(y_test, test_preds, zero_division=0):.6f}")
print(f"  F1:        {f1_score(y_test, test_preds, zero_division=0):.6f}")
try: print(f"  AUC-ROC:   {roc_auc_score(y_test, test_probs):.6f}")
except: print("  AUC-ROC:   N/A")
print("\n" + classification_report(y_test, test_preds, target_names=['Down','Up']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cm = confusion_matrix(y_test, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Down','Up'], yticklabels=['Down','Up'], ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Test — Confusion Matrix')

try:
    fpr, tpr, _ = roc_curve(y_test, test_probs)
    auc = roc_auc_score(y_test, test_probs)
    axes[1].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {auc:.4f}')
    axes[1].plot([0,1],[0,1], 'navy', lw=1, ls='--', label='Random')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].set_title('Test — ROC Curve'); axes[1].legend()
except ValueError:
    axes[1].text(0.5, 0.5, 'ROC unavailable', ha='center')

plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
def _auc(yt,yp):
    try: return roc_auc_score(yt,yp)
    except: return 0.5

comp = pd.DataFrame([
    {'Model': 'BiRNN + Attention Default CV',
      **{k: def_m[k] for k in ['accuracy','precision','recall','f1','auc_roc']}},
    {'Model': 'BiRNN + Attention Optuna CV',
      **{k: opt_m[k] for k in ['accuracy','precision','recall','f1','auc_roc']}},
    {'Model': 'BiRNN + Attention Final Test',
      'accuracy': accuracy_score(y_test, test_preds),
      'precision': precision_score(y_test, test_preds, zero_division=0),
      'recall': recall_score(y_test, test_preds, zero_division=0),
      'f1': f1_score(y_test, test_preds, zero_division=0),
      'auc_roc': _auc(y_test, test_probs)},
]).set_index('Model')

comp.style.highlight_max(subset=['accuracy','f1','auc_roc'], color='#d4edda').format('{:.6f}')